# Day 1.6 — Represent the Agent Loop with LangGraph

We already understand the loop. LangGraph now gives it an explicit state-and-transition structure:

```text
START → model ──final──→ END
          └─tool request→ tools → model
          └─step limit──→ limit → END
```

LangGraph is not the agent's intelligence and does not replace the provider API. It organizes execution.

## Before you begin

### Learning outcomes

Represent the same loop as state, nodes, edges, and conditional routing.

Architecture reference: [D05](../../diagrams/source/day_01.md).

### Expected observation

The final state contains the message history and a terminal route.


## Concept briefing

## Workflow or agent?

Not every problem needs an agent. Use ordinary code or a deterministic workflow when the
steps and decision rules are known. Use a hybrid workflow when most steps are fixed but
one bounded judgment benefits from a model. Consider an agent when the next action cannot
be fully predetermined, the action set is small, failures are containable, and success
can be evaluated.

Ask:

1. Are the steps known in advance?
2. Can normal code make the decision reliably?
3. Does the model genuinely add judgment rather than decoration?
4. What is the consequence of a wrong action?
5. Is there a strict step and tool boundary?
6. Can we observe and evaluate the result?

If these questions have weak answers, the correct design is often a workflow, not an
agent.


## Learning objectives

Explain node, edge, conditional edge, and state; map the manual loop to a graph; and confirm that graph execution preserves the same model/tool responsibilities.

In [ ]:
# Install once if needed:
# %pip install -q langgraph

import sys
from pathlib import Path
from dotenv import load_dotenv

here = Path.cwd().resolve()
candidates = [here, here / "day_01_model_tools_agent", here.parent]
project_root = next(path for path in candidates if (path / "src" / "research_agent").exists())
sys.path.insert(0, str(project_root / "src"))
load_dotenv()

from research_agent.agent import SYSTEM_MESSAGE
from research_agent.graph import build_graph
from research_agent.providers import OpenRouterProvider
from research_agent.schemas import Message
from research_agent.tools import default_tool_registry

## State is application-owned data

Our graph state carries messages, current step count, maximum steps, final validated response, and error. Nodes read state and return updates.

In [ ]:
initial_state = {
    "messages": [
        Message(role="system", content=SYSTEM_MESSAGE),
        Message(role="user", content="Explain an AI tool using notes and calculate 12 * 7."),
    ],
    "steps": 0,
    "max_steps": 5,
    "final_response": None,
    "error": None,
}
initial_state

## Compile the graph

The implementation uses `StateGraph`, Python node functions, and conditional edges. Provider calls remain plain calls through `OpenRouterProvider`; no LangChain agent, chain, LCEL, or memory abstraction is used.

In [ ]:
graph = build_graph(OpenRouterProvider(), default_tool_registry())
print(graph.get_graph().draw_mermaid())

## Invoke the graph and inspect final state

In [ ]:
final_state = graph.invoke(initial_state)
print("steps:", final_state["steps"])
print("error:", final_state["error"])
print(final_state["final_response"].model_dump_json(indent=2) if final_state["final_response"] else "No final response")

## Observe the route from messages

In [ ]:
for message in final_state["messages"]:
    requests = [call.name for call in message.tool_calls]
    print(message.role, requests, message.content[:100])

## Compare manual loop and graph

| Manual loop | Graph |
|---|---|
| `for` iteration | model node visited repeatedly |
| `if tool_calls` | conditional edge |
| execute functions | tools node |
| return result | edge to END |
| step counter | state field and limit route |

The architecture is the same; the representation is more explicit.

## Exercise and checkpoint

Set `max_steps` to 1 and inspect the limit result. Then read `src/research_agent/graph.py` and label its model node, tools node, routing function, and edges.

We use LangGraph because later projects need state, branching, interrupts, and checkpoints—not because it magically creates an agent.

## Your turn

Change one routing condition to a safe failure and inspect the final state.

## Recap

LangGraph represents orchestration; it does not replace tools, policy, or evaluation.
